# Module 8: Window Functions

Window functions are one of the **most asked topics in Spark interviews**. They let you compute values across a set of rows *related to the current row* — without collapsing the rows like `groupBy` does.

| `groupBy` + `agg` | Window Function |
|---|---|
| Collapses rows into one per group | Keeps all rows, adds a new column |
| `avg(salary)` gives one row per city | `avg(salary).over(window)` adds the city average to every row |

In this module you'll learn:
1. **Window specs** — `partitionBy`, `orderBy`
2. **Ranking** — `row_number`, `rank`, `dense_rank`
3. **Analytic functions** — `lag`, `lead`
4. **Aggregate windows** — running totals, moving averages
5. **Frame specs** — `rowsBetween`, `rangeBetween`

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, row_number, rank, dense_rank, lag, lead,
    sum, avg, count, round, datediff, lit, month, year
)

spark = SparkSession.builder \
    .appName("Module 08 - Window Functions") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

employees.show(5)
sales.show(5)

---
## Concept 1: Window Specs — Defining the Window

A **WindowSpec** defines:
- `partitionBy(col)` — which groups to compute over (like `GROUP BY`, but rows aren't collapsed)
- `orderBy(col)` — the sort order within each partition

Think of it as: "For each row, look at the other rows in my partition (sorted this way) and compute something."

```python
window = Window.partitionBy("city").orderBy(col("salary").desc())
```

This says: partition employees by city, and within each city sort by salary descending.

In [ ]:
# Define a window: partition by city, order by salary descending
city_salary_window = Window.partitionBy("city").orderBy(col("salary").desc())

# Add row_number — this is a ranking function (covered next)
# For now, just see how the window works
employees.withColumn(
    "rank_in_city", row_number().over(city_salary_window)
).select("city", "name", "salary", "rank_in_city").show(30)

Notice: every row is still there. But each employee now has a rank *within their city*. That's the power of window functions — you compute per-group values without losing rows.

---
## Concept 2: Ranking Functions — row_number, rank, dense_rank

Three ranking functions — they differ in how they handle **ties**:

| Function | Ties | Gaps | Example (scores: 100, 90, 90, 80) |
|----------|------|------|------------------------------------|
| `row_number()` | No ties (arbitrary) | No gaps | 1, 2, 3, 4 |
| `rank()` | Same rank for ties | Gaps after ties | 1, 2, 2, 4 |
| `dense_rank()` | Same rank for ties | No gaps | 1, 2, 2, 3 |

**Interview tip**: `row_number()` is used when you need exactly one row per position (e.g., "get the top earner per city"). `rank()` and `dense_rank()` are used when ties matter.

In [ ]:
# Compare all three ranking functions
window_by_city = Window.partitionBy("city").orderBy(col("salary").desc())

ranked = employees.select(
    "city", "name", "salary",
    row_number().over(window_by_city).alias("row_num"),
    rank().over(window_by_city).alias("rank"),
    dense_rank().over(window_by_city).alias("dense_rank")
)

ranked.show(30)

In [ ]:
# Common interview question: Get the highest-paid employee in each city
top_per_city = ranked.filter(col("row_num") == 1)
top_per_city.show()

#### Try It: Rank Sales by Amount per Region

1. Define a window partitioned by `region`, ordered by `amount` descending
2. Add `row_number`, `rank`, and `dense_rank` columns
3. Show the top 3 sales per region (filter `row_num <= 3`)

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
region_window = Window.partitionBy("region").orderBy(col("amount").desc())

sales_ranked = sales.select(
    "region", "product", "amount", "quantity",
    row_number().over(region_window).alias("row_num"),
    rank().over(region_window).alias("rank"),
    dense_rank().over(region_window).alias("dense_rank")
)

sales_ranked.filter(col("row_num") <= 3).orderBy("region", "row_num").show()

---
## Concept 3: Analytic Functions — lag and lead

`lag(col, n)` — value from `n` rows **before** the current row
`lead(col, n)` — value from `n` rows **after** the current row

These are essential for **comparing adjacent rows**: previous month's sales, day-over-day change, etc.

If there's no previous/next row, the result is `null` (or you can set a default).

In [ ]:
# For each employee's sales (ordered by date), show previous and next sale amounts
emp_sales_window = Window.partitionBy("employee_id").orderBy("sale_date")

sales_with_lag = sales.select(
    "employee_id", "sale_date", "amount",
    lag("amount", 1).over(emp_sales_window).alias("prev_amount"),
    lead("amount", 1).over(emp_sales_window).alias("next_amount")
)

# Show for employee 4 as an example
sales_with_lag.filter(col("employee_id") == 4).show()

In [ ]:
# Calculate sale-over-sale change
sales_change = sales.withColumn(
    "prev_amount", lag("amount", 1).over(emp_sales_window)
).withColumn(
    "change", round(col("amount") - col("prev_amount"), 2)
).withColumn(
    "pct_change", round((col("amount") - col("prev_amount")) / col("prev_amount") * 100, 2)
)

sales_change.filter(col("employee_id") == 4) \
    .select("employee_id", "sale_date", "amount", "prev_amount", "change", "pct_change").show()

#### Try It: Compare Sales to Previous Sale by Region

1. Define a window partitioned by `region`, ordered by `sale_date`
2. Add `prev_amount` using `lag`
3. Add `change` column (current amount - previous amount)
4. Show the first 15 rows for the `"West"` region

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
region_date_window = Window.partitionBy("region").orderBy("sale_date")

region_change = sales.withColumn(
    "prev_amount", lag("amount", 1).over(region_date_window)
).withColumn(
    "change", round(col("amount") - col("prev_amount"), 2)
)

region_change.filter(col("region") == "West") \
    .select("region", "sale_date", "product", "amount", "prev_amount", "change").show(15)

---
## Concept 4: Aggregate Window Functions — Running Totals

You can use `sum()`, `avg()`, `count()`, `min()`, `max()` as window functions too. Combined with `orderBy`, they create **running (cumulative) aggregates**.

- `sum("amount").over(window)` — running total
- `avg("amount").over(window)` — running average
- `count("*").over(window)` — running count

Without `orderBy`, they compute the aggregate over the **entire partition** (same value for every row in the group — like adding the group's total as a column).

In [ ]:
# Running total of sales amount by region, ordered by sale_date
running_window = Window.partitionBy("region").orderBy("sale_date")

sales_running = sales.select(
    "region", "sale_date", "amount",
    round(sum("amount").over(running_window), 2).alias("running_total"),
    round(avg("amount").over(running_window), 2).alias("running_avg"),
    count("*").over(running_window).alias("running_count")
)

sales_running.filter(col("region") == "West").show(15)

In [ ]:
# Without orderBy — gives the TOTAL for the entire partition (same value every row)
partition_only = Window.partitionBy("region")

sales.select(
    "region", "sale_date", "amount",
    round(sum("amount").over(partition_only), 2).alias("region_total"),
    round(avg("amount").over(partition_only), 2).alias("region_avg")
).filter(col("region") == "West").show(10)

Notice the difference:
- **With `orderBy`**: running total grows row by row
- **Without `orderBy`**: every row shows the same partition-wide total

Both are useful. Use partition-only windows when you want to add a group statistic to each row (e.g., "what percentage of my region's total is this sale?").

In [ ]:
# Percentage of region total — a classic interview question
sales.select(
    "region", "product", "amount",
    round(sum("amount").over(partition_only), 2).alias("region_total"),
    round(col("amount") / sum("amount").over(partition_only) * 100, 2).alias("pct_of_region")
).filter(col("region") == "West").show(10)

#### Try It: Running Total by Employee

1. Define a window partitioned by `employee_id`, ordered by `sale_date`
2. Add `running_total` (cumulative sum of `amount`)
3. Add `sale_number` (running count)
4. Show results for `employee_id == 4`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
emp_running_window = Window.partitionBy("employee_id").orderBy("sale_date")

emp_running = sales.select(
    "employee_id", "sale_date", "product", "amount",
    round(sum("amount").over(emp_running_window), 2).alias("running_total"),
    count("*").over(emp_running_window).alias("sale_number")
)

emp_running.filter(col("employee_id") == 4).show()

---
## Concept 5: Frame Specifications — Sliding Windows

By default, an ordered window includes all rows **from the start of the partition to the current row**. You can change this with frame specs:

- `rowsBetween(start, end)` — a fixed number of rows before/after
- `rangeBetween(start, end)` — based on the value difference in the `orderBy` column

Special constants:
- `Window.unboundedPreceding` — from the very first row
- `Window.unboundedFollowing` — to the very last row
- `Window.currentRow` — the current row (value `0`)

**Common use case**: moving averages.

In [ ]:
# 3-row moving average of sale amount within each region
moving_avg_window = Window.partitionBy("region").orderBy("sale_date") \
    .rowsBetween(-2, Window.currentRow)  # current row + 2 previous rows

sales_ma = sales.select(
    "region", "sale_date", "amount",
    round(avg("amount").over(moving_avg_window), 2).alias("moving_avg_3")
)

sales_ma.filter(col("region") == "West").show(15)

In [ ]:
# Full partition aggregate — sum from first to last row in partition
full_window = Window.partitionBy("region").orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

sales.select(
    "region", "sale_date", "amount",
    round(sum("amount").over(full_window), 2).alias("region_total")
).filter(col("region") == "West").show(10)

#### Try It: 5-Row Moving Average

1. Create a window partitioned by `region`, ordered by `sale_date`, with a frame of 4 preceding rows + current row (5-row window)
2. Compute the moving average of `amount`
3. Show results for the `"East"` region

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
ma5_window = Window.partitionBy("region").orderBy("sale_date") \
    .rowsBetween(-4, Window.currentRow)

sales.select(
    "region", "sale_date", "amount",
    round(avg("amount").over(ma5_window), 2).alias("moving_avg_5")
).filter(col("region") == "East").show(15)

---
## Concept 6: Window Functions in SQL

Everything above works in Spark SQL too. The syntax is the standard SQL `OVER` clause. Interviewers often ask window questions in SQL form.

In [ ]:
# Register as temp views for SQL
employees.createOrReplaceTempView("employees")
sales.createOrReplaceTempView("sales")

# Ranking in SQL
spark.sql("""
    SELECT city, name, salary,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY salary DESC) AS row_num,
           RANK()       OVER (PARTITION BY city ORDER BY salary DESC) AS rank,
           DENSE_RANK() OVER (PARTITION BY city ORDER BY salary DESC) AS dense_rank
    FROM employees
    ORDER BY city, row_num
""").show(30)

In [ ]:
# Running total and lag in SQL
spark.sql("""
    SELECT region, sale_date, amount,
           SUM(amount) OVER (PARTITION BY region ORDER BY sale_date) AS running_total,
           LAG(amount, 1) OVER (PARTITION BY region ORDER BY sale_date) AS prev_amount
    FROM sales
    WHERE region = 'West'
    ORDER BY sale_date
""").show(15)

---
## Capstone Exercise

Build a comprehensive sales analysis using window functions:

1. Join `sales` with `employees` (on `employee_id`) to get employee `name`
2. For each employee, ordered by `sale_date`, add:
   - `sale_number` — running count of sales (1st sale, 2nd sale, etc.)
   - `running_total` — cumulative sum of `amount`
   - `prev_amount` — previous sale amount using `lag`
   - `change` — difference from previous sale
3. Also add a `rank_in_region` column — rank each sale by `amount` descending within `region`
4. Show the results for one employee of your choice

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---
sales_with_name = sales.join(
    employees.select("employee_id", "name"),
    on="employee_id"
)

# Window for per-employee running stats
emp_window = Window.partitionBy("employee_id").orderBy("sale_date")

# Window for ranking within region
region_rank_window = Window.partitionBy("region").orderBy(col("amount").desc())

capstone = sales_with_name \
    .withColumn("sale_number", count("*").over(emp_window)) \
    .withColumn("running_total", round(sum("amount").over(emp_window), 2)) \
    .withColumn("prev_amount", lag("amount", 1).over(emp_window)) \
    .withColumn("change", round(col("amount") - col("prev_amount"), 2)) \
    .withColumn("rank_in_region", rank().over(region_rank_window))

capstone.filter(col("employee_id") == 4) \
    .select("name", "sale_date", "region", "amount",
            "sale_number", "running_total", "prev_amount", "change", "rank_in_region") \
    .show()

In [ ]:
spark.stop()

---
## What You Learned

- **Window specs** define partitions and ordering without collapsing rows
- **`row_number`** — unique sequential rank (no ties)
- **`rank`** — allows ties, leaves gaps; **`dense_rank`** — allows ties, no gaps
- **`lag`/`lead`** — access previous/next row values for comparisons
- **Aggregate windows** — `sum`, `avg`, `count` over ordered windows give running totals
- **Frame specs** — `rowsBetween` controls the sliding window size for moving averages
- All window functions work in both DataFrame API and SQL

### Window Functions Quick Reference

| Function | What it does | Needs orderBy? |
|----------|-------------|----------------|
| `row_number()` | Unique sequential number | Yes |
| `rank()` | Rank with gaps on ties | Yes |
| `dense_rank()` | Rank without gaps on ties | Yes |
| `lag(col, n)` | Value n rows before | Yes |
| `lead(col, n)` | Value n rows after | Yes |
| `sum(col)` | Running/partition total | Optional |
| `avg(col)` | Running/partition average | Optional |
| `count(col)` | Running/partition count | Optional |

Next: **Module 9 — Spark MLlib**